# 03 · Build the DuckDB warehouse and answer the business questions in SQL

This notebook only *runs* SQL — all the logic lives in the `sql/` folder so it can be read
(and run) on its own:

| File | Does |
|---|---|
| `sql/01_load_raw.sql` | Copies the source files (3 real, 2 simulated) into the warehouse unchanged |
| `sql/02_build_tables.sql` | Star schema + summary tables + renewal / provision analysis |
| `sql/03_quality_checks.sql` | 23 data-quality checks — the build **stops** if an error-level check fails |
| `sql/04_analysis_queries.sql` | The business questions (Q1–Q12) and the check against real data (Q13–Q15) |
| `sql/05_export_powerbi.sql` | CSV exports for Power BI |

**DuckDB** is a database stored in a single file (`data/warehouse.duckdb`) — no server to
install. The SQL is standard and would run on Postgres or Snowflake with minor changes.

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import duckdb
import pandas as pd
import yaml

from src.logger import get_logger

log = get_logger("03_build_warehouse")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

con = duckdb.connect("data/warehouse.duckdb")


def run_sql_file(path):
    """Run every statement in a .sql file."""
    con.execute(open(path, encoding="utf-8").read())
    log.info(f"Ran {path}")

## 1. Load the config values the SQL needs
Spreads, LGDs and thresholds come from `config/pipeline.yml`, loaded into a small table
`raw.assumptions`. That way **no number is typed directly into the SQL**.

In [2]:
config = yaml.safe_load(open("config/pipeline.yml"))

assumptions = pd.DataFrame(
    [
        ("fixed_spread_over_goc_5yr", config["risk"]["mortgage"]["fixed_spread_over_goc_5yr"]),
        ("fixed_rate_floor", config["risk"]["mortgage"]["fixed_rate_floor"]),
        ("rate_increase_per_pp", config["hazard_drivers"]["rate_increase_per_pp"]),
        ("severe_payment_shock_pct", config["analysis"]["severe_payment_shock_pct"]),
    ]
    + [(f"lgd_{product}", settings["lgd"]) for product, settings in config["risk"].items()],
    columns=["name", "value"],
)

con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE OR REPLACE TABLE raw.assumptions AS SELECT * FROM assumptions")
assumptions

,name,value
0,fixed_spread_over_goc_5yr,1.40
1,fixed_rate_floor,1.35
2,rate_increase_per_pp,0.25
3,severe_payment_shock_pct,25.00
4,lgd_mortgage,0.10
5,lgd_heloc,0.25
6,lgd_auto,0.45
7,lgd_card,0.85


## 2. Load raw data and build the tables

In [3]:
run_sql_file("sql/01_load_raw.sql")
run_sql_file("sql/02_build_tables.sql")

tables = con.sql("""
    SELECT schema_name, table_name, estimated_size AS row_count
    FROM duckdb_tables()
    ORDER BY schema_name, table_name
""").df()
tables

2026-10-17 19:02:25,871 | 03_build_warehouse | INFO | Ran sql/01_load_raw.sql


2026-10-17 19:02:27,571 | 03_build_warehouse | INFO | Ran sql/02_build_tables.sql


,schema_name,table_name,row_count
0,mart,agg_portfolio_monthly,8290
1,mart,agg_vintage,46
2,mart,arrears_validation,336
3,mart,dim_credit_band,5
4,mart,dim_loan,50000
5,mart,dim_month,156
6,mart,dim_product,4
7,mart,dim_province,10
8,mart,fact_loan_month,1688653
9,mart,fact_province_month,1410


## 3. Data quality gate
Each check counts rows that **break** a rule. If any `error` check fails, this cell raises an
error and the rest of the notebook doesn't run — bad data never reaches the dashboard.

In [4]:
run_sql_file("sql/03_quality_checks.sql")
checks = con.sql("SELECT dimension, check_name, severity, failed_rows, result FROM quality.check_results").df()

failed_errors = checks[(checks["result"] == "FAIL") & (checks["severity"] == "error")]
if len(failed_errors) > 0:
    raise ValueError(f"{len(failed_errors)} data quality checks failed:\n{failed_errors}")

log.info(f"Data quality: {(checks['result'] == 'PASS').sum()} of {len(checks)} checks passed")
checks

2026-10-17 19:02:28,076 | 03_build_warehouse | INFO | Ran sql/03_quality_checks.sql


2026-10-17 19:02:28,081 | 03_build_warehouse | INFO | Data quality: 23 of 23 checks passed


,dimension,check_name,severity,failed_rows,result
0,completeness,"Every loan has a product, province and credit ...",error,0,PASS
1,completeness,Every month in the data has all four real rates,error,0,PASS
2,completeness,Every renewing mortgage has old and new payment,error,0,PASS
3,uniqueness,loan_id is unique in dim_loan,error,0,PASS
4,uniqueness,One row per loan per month in fact_loan_month,error,0,PASS
5,uniqueness,One row per month in dim_month,error,0,PASS
6,validity,Credit scores between 300 and 900,error,0,PASS
7,validity,Balances are not negative,error,0,PASS
8,validity,Status is one of the six allowed values,error,0,PASS
9,validity,Interest rates between 0% and 30%,error,0,PASS


## 4. The business questions
Each query lives in `sql/04_analysis_queries.sql`. `ask("Q1")` runs one of them.

In [5]:
# Split the file into its queries and file them under "Q1", "Q2", ...
queries = {}
for statement in open("sql/04_analysis_queries.sql", encoding="utf-8").read().split(";"):
    for line in statement.splitlines():
        if line.startswith("-- Q"):
            key = line.split(".")[0].replace("-- ", "")   # "-- Q1. How big..." -> "Q1"
            queries[key] = (line.replace("-- ", ""), statement)
            break


def ask(key):
    title, sql = queries[key]
    print(title)
    return con.sql(sql).df()


print(f"{len(queries)} questions loaded: {', '.join(queries)}")

15 questions loaded: Q1, Q2, Q3, Q4, Q5, Q6, Q7, Q8, Q9, Q10, Q11, Q12, Q13, Q14, Q15


### Q12 · The headline numbers first
These are the five KPI cards on the dashboard. Use this row to check your Power BI cards.

In [6]:
ask("Q12")

Q12. Headline numbers (the KPI cards on the dashboard, and the README summary)


,total_exposure_bn,delinquency_rate_pct,impaired_rate_pct,renewing_mortgages,renewing_balance_bn,weighted_avg_shock_pct,at_risk_balance_m,added_annual_payments_m,incremental_provision_cad,borrowers_still_to_call
0,8.4,0.58,0.1,6716,2.7,15.2,244.9,28.3,211222.0,2797


### Q1 · The renewal wave, year by year

In [7]:
ask("Q1")

Q1. How big is the renewal wave, and how hard does it hit each year?


,renewal_year,renewing_loans,already_renewed,renewing_balance_m,weighted_avg_shock_pct,added_annual_payments_m,at_risk_balance_m
0,2025,2189,2189,852.8,19.8,11.22,164.9
1,2026,2250,1716,900.8,20.2,12.46,79.9
2,2027,2277,0,950.0,6.3,4.61,0.0


**Reading it:** 2025 and 2026 renewals face a ~20% higher payment (balance-weighted). 2027 is
much milder (~6%) because those terms started in 2022, after rates had already risen.
All 2025 renewals have already happened by Sep 2026; most of 2026 has; none of 2027 has — those
are the borrowers the bank can still reach before the shock.

### Q2 · How big is the shock?

In [8]:
ask("Q2")

Q2. What does the shock look like? Loans and balance by shock band


,shock_band,loans,balance_m,pct_of_renewing_balance,avg_monthly_increase_cad
0,0-10%,1707,726.5,26.9,103.0
1,10-25%,4445,1732.3,64.1,420.0
2,Over 25%,564,244.9,9.1,561.0


### Q3 · What actually drives the shock: *when* the term started

In [9]:
ask("Q3")

Q3. Which year did the current term start? (this is what really drives the shock)


,term_started,loans,median_old_rate,median_new_rate,median_shock_pct,median_monthly_increase_cad
0,2020,2189,1.82,4.20,19.1,405.0
1,2021,2250,2.28,4.56,20.2,450.0
2,2022,2277,4.38,4.94,4.2,120.0


**Reading it:** a term locked in 2020 at a median 1.82% renews at 4.20%; 2021 at 2.28% → 4.56%.
Terms locked in 2022 started at 4.38%, so they barely move. **Timing, not borrower quality,
decides who gets shocked.**

### Q4 · Provinces

In [10]:
ask("Q4")

Q4. Which provinces carry the most renewal risk? Size (total provision) and intensity


,province_name,real_arrears_factor,avg_unemployment_pct,renewing_loans,renewing_balance_m,weighted_avg_shock_pct,incremental_provision_cad,pct_of_total_provision,provision_per_1m_balance
0,Ontario,0.89,7.0,2983,1190.3,15.2,82036.0,38.8,69.0
1,Alberta,1.38,6.8,769,311.1,15.3,34229.0,16.2,110.0
2,Quebec,0.78,5.3,1247,512.5,15.4,32350.0,15.3,63.0
3,British Columbia,0.85,5.9,977,386.1,15.3,26194.0,12.4,68.0
4,Saskatchewan,2.58,5.3,155,61.1,16.3,12987.0,6.1,213.0
5,Nova Scotia,1.28,6.5,189,79.9,14.9,7828.0,3.7,98.0
6,Manitoba,1.49,5.4,155,64.2,14.3,6291.0,3.0,98.0
7,New Brunswick,1.28,7.0,124,49.9,13.8,4478.0,2.1,90.0
8,Newfoundland and Labrador,1.28,9.8,99,40.7,14.8,3956.0,1.9,97.0
9,Prince Edward Island,1.28,7.7,18,7.9,13.9,874.0,0.4,111.0


**Reading it:** two different answers to "which province is riskiest?":
- **By size**, Ontario holds the most extra provision (38.8%), because it has the most mortgages.
- **By intensity** (provision per $1M of renewing balance), **Saskatchewan is highest at $213**,
  about three times Quebec's $63, because real Saskatchewan mortgages fall 90+ days behind
  2.58× as often as the Canadian average. Alberta and Manitoba are also well above average.

Note the payment shock itself is about the same everywhere (~14–16%): the difference between
provinces is how likely a borrower is to fall behind, which comes from the real arrears data.
Unemployment doesn't explain it: Saskatchewan's unemployment (5.3%) is among the lowest.

### Q5 · Does the shock land on weaker borrowers?
**Lift** = a band's share of the high-shock group ÷ its share of all renewals.
Lift > 1 means over-represented.

In [11]:
ask("Q5")

Q5. Does payment shock fall on weaker borrowers? Compare each credit band's share of


,credit_band,all_renewals,high_shock,pct_of_renewals,pct_of_high_shock,lift
0,Below 600,302,84,4.5,3.9,0.86
1,600-659,1015,313,15.1,14.4,0.96
2,660-719,2030,665,30.2,30.7,1.01
3,720-779,1997,657,29.7,30.3,1.02
4,780+,1372,449,20.4,20.7,1.01


**Reading it:** every band's lift is close to 1 (0.86–1.02). The shock is spread evenly across
credit quality — it depends on when the rate was locked, not on who the borrower is. The *risk*
is where the shock meets a weak credit score, which is what the provision (Q6) and the call list
(Q10) pick up.

### Q6 · Extra provision by credit band

In [12]:
ask("Q6")

Q6. Where should the extra provision be held? By credit band.


,credit_band,renewing_loans,base_pd_12m_pct,stressed_pd_12m_pct,expected_loss_before_cad,expected_loss_after_cad,incremental_provision_cad
0,Below 600,302,0.547,0.785,68285.0,97394.0,29108.0
1,600-659,1015,0.285,0.409,118488.0,168914.0,50427.0
2,660-719,2030,0.198,0.286,160404.0,229910.0,69506.0
3,720-779,1997,0.122,0.177,97314.0,140075.0,42762.0
4,780+,1372,0.079,0.115,44313.0,63732.0,19419.0


**Reading it:** the mortgage default probability falls steadily with credit score (Q8), so
"Below 600" borrowers carry the most extra provision *per loan*. (Each borrower's PD is also
scaled by their region's real arrears factor, which is why the band averages here differ slightly from Q8.) The 660–719 band holds the
most provision *overall* simply because it has the most loans.

**Caveat:** each band has only 7–26 mortgage defaults, so these PDs carry real sampling noise.
A bank would smooth them with a model — notebook 05 fits one.

### Q7 · Portfolio health today, by product

In [13]:
ask("Q7")

Q7. Headline portfolio KPIs in the latest month, by product


,product_name,open_loans,exposure_m,delinquency_rate_pct,impaired_rate_pct
0,Mortgage,17367,7428.8,0.53,0.08
1,HELOC,4863,655.1,0.84,0.24
2,Auto loan,6232,142.1,1.21,0.29
3,Credit card,16978,175.3,1.44,0.50


### Q8 · Default rates by product and credit band
Sanity check: secured lending should default least, unsecured most
(mortgage < HELOC < auto < card), and lower credit bands should default more.

In [14]:
ask("Q8")

Q8. Default rates by product and credit band (the PD table behind the provision)


,product_name,credit_band,loans_in_cohort,defaulted_loans,default_rate_42m_pct,pd_12m_pct
0,Mortgage,Below 600,523,10,1.91,0.550
1,Mortgage,600-659,1880,19,1.01,0.290
2,Mortgage,660-719,3758,26,0.69,0.198
3,Mortgage,720-779,3722,16,0.43,0.123
4,Mortgage,780+,2526,7,0.28,0.079
5,HELOC,Below 600,177,4,2.26,0.651
6,HELOC,600-659,557,14,2.51,0.725
7,HELOC,660-719,1026,12,1.17,0.336
8,HELOC,720-779,1049,12,1.14,0.328
9,HELOC,780+,680,6,0.88,0.253


### Q9 · Did the 2023 rate spike show up in HELOCs?

In [15]:
ask("Q9")

Q9. Did rising rates show up in HELOCs (their rate floats with prime)? HELOC delinquency vs prime, by quarter


,year,quarter,avg_prime_rate,heloc_delinquency_pct
0,2023,2,6.76,0.48
1,2023,3,7.16,0.74
2,2023,4,7.20,0.76
3,2024,1,7.20,0.88
4,2024,2,7.14,0.70
5,2024,3,6.70,0.61
6,2024,4,6.00,0.57
7,2025,1,5.24,0.50
8,2025,2,4.95,0.63
9,2025,3,4.93,0.69


**Reading it:** HELOC rates float with prime. As prime climbed to 7.2% through 2023, HELOC
delinquency rose from 0.48% to a peak of 0.88% (Q1 2024), then eased to ~0.5–0.7% as prime fell
back to 4.45%. This is the real rate path feeding through the simulation.

### Q10 · The call list: who to contact first
Only borrowers who **haven't renewed yet** and **haven't defaulted**, ranked by how much extra
loss their renewal adds. Low scores, big rate jumps and high-arrears regions (Alberta,
Saskatchewan) rise to the top.

In [16]:
ask("Q10")

Q10. Who do we call first? Top 15 borrowers who haven't renewed yet


,outreach_priority,loan_id,province_name,credit_score,renewal_month,old_rate,new_rate,monthly_increase_cad,payment_shock_pct,incremental_provision_cad
0,1,30258,Alberta,586,2026-10-01,2.83,5.14,875.0,22.26,315.0
1,2,43834,Alberta,586,2026-11-01,3.04,5.14,726.0,19.99,259.0
2,3,12446,Saskatchewan,637,2026-10-01,2.75,5.06,619.0,22.33,221.0
3,4,39966,Ontario,595,2026-12-01,2.86,5.13,766.0,21.80,177.0
4,5,4495,Ontario,533,2027-02-01,3.37,5.22,744.0,17.21,170.0
5,6,44972,Nova Scotia,588,2026-12-01,2.87,5.14,473.0,16.60,166.0
6,7,45179,Saskatchewan,562,2026-11-01,3.07,5.18,243.0,19.96,163.0
7,8,15781,British Columbia,516,2026-12-01,2.98,5.24,701.0,16.55,163.0
8,9,5169,Alberta,631,2026-10-01,2.76,5.07,850.0,22.32,162.0
9,10,16513,Alberta,564,2027-02-01,3.33,5.17,428.0,13.23,159.0


### Q11 · Data quality summary

In [17]:
ask("Q11")

Q11. Data quality summary


,dimension,checks,passed,failed
0,completeness,5,5,0
1,consistency,4,4,0
2,reconciliation,4,4,0
3,uniqueness,3,3,0
4,validity,7,7,0


## 5. Validation: does the simulation behave like the REAL market?
We drive the simulation with real unemployment (Statistics Canada) and check it against a
**different** real source: the Canadian Bankers Association's count of mortgages 90+ days
behind. Using one real source to drive and another to check keeps the test honest.

In [18]:
ask("Q15")

Q15. Validation in one row: how closely do simulated and real arrears move together?


,months_compared,real_avg_arrears_pct,simulated_avg_arrears_pct,simulated_to_real_ratio,sim_vs_real_monthly_corr,sim_vs_real_quarterly_corr,real_arrears_vs_unemployment_over_time,real_arrears_vs_unemployment_across_regions
0,37,0.219,0.089,0.41,0.19,0.41,0.69,-0.29


**Reading it:**
- **Real vs real first.** Real arrears and real unemployment moved together over time
  (correlation 0.69) but not across regions (−0.29). That's why the simulation uses unemployment
  for *when* stress rises and real regional arrears for *where* it's high.
- **Level:** the simulation's arrears are about **0.41×** the real level (0.089% vs 0.219%). The
  main reason is structural: a real mortgage can stay 90+ days behind for many months while the
  bank goes through the legal process to sell the home, but our model writes a loan off after
  120 days, so fewer loans sit in "arrears" at any moment. Matching the level would mean
  modelling foreclosure timelines, which is out of scope; we report the gap instead.
- **Trend:** the quarterly correlation is **0.41**: the simulation rises with the real data, but
  loosely. With ~17,000 simulated mortgages only ~15 are 90+ days behind in a typical month, so
  month-to-month noise is large.

In [19]:
ask("Q14")

Q14. Does the simulation follow the REAL national arrears trend? By quarter.


,year,quarter,real_arrears_pct,simulated_arrears_pct
0,2023,3,0.156,0.109
1,2023,4,0.172,0.060
2,2024,1,0.185,0.073
3,2024,2,0.188,0.100
4,2024,3,0.200,0.077
5,2024,4,0.211,0.075
6,2025,1,0.223,0.068
7,2025,2,0.223,0.099
8,2025,3,0.236,0.122
9,2025,4,0.252,0.078


In [20]:
ask("Q13")

Q13. Does the simulation match REAL mortgage arrears by region?


,region,real_unemployment_pct,real_arrears_pct,simulated_arrears_pct,real_rank,simulated_rank
0,Saskatchewan,5.3,0.549,0.273,1,1
1,Manitoba,5.5,0.320,0.165,2,2
2,Alberta,6.9,0.291,0.124,3,3
3,Atlantic,7.5,0.276,0.115,4,4
4,Ontario,7.1,0.198,0.074,5,6
5,British Columbia,5.9,0.187,0.100,6,5
6,Quebec,5.4,0.171,0.056,7,7


**Reading it:** the regional ranking matches the real one for 5 of 7 regions (Ontario and BC swap
because of random noise). This is **by design, not a test**: the regional factor was calibrated
to these same real arrears. The genuine test is the time trend above.

## 6. Export for Power BI

In [21]:
os.makedirs("powerbi/data", exist_ok=True)
run_sql_file("sql/05_export_powerbi.sql")

exports = pd.DataFrame({"file": sorted(os.listdir("powerbi/data"))})
exports["size_kb"] = [round(os.path.getsize(f"powerbi/data/{f}") / 1024, 1) for f in exports["file"]]
exports

2026-10-17 19:02:28,933 | 03_build_warehouse | INFO | Ran sql/05_export_powerbi.sql


,file,size_kb
0,agg_portfolio_monthly.csv,808.3
1,agg_vintage.csv,3.7
2,arrears_validation.csv,31.3
3,data_quality_checks.csv,2.3
4,dim_credit_band.csv,0.1
5,dim_month.csv,15.7
6,dim_product.csv,0.2
7,dim_province.csv,0.6
8,fact_province_month.csv,92.0
9,fact_renewal.csv,2118.3


In [22]:
# Close the connection so other notebooks (and DuckDB tools) can open the warehouse file
con.close()
log.info("Warehouse built and exported")

2026-10-17 19:02:29,009 | 03_build_warehouse | INFO | Warehouse built and exported
